In [1]:
import os, glob
import numpy as np, pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from tensorboard.util.tensor_util import make_ndarray

log_root = r"C:\Users\adria\VSCode_Projects\machineLearning\DW-DL\code\logs"

def series(d, tag):
    if not os.path.isdir(d): return None
    ea = EventAccumulator(d, size_guidance={"tensors": 0, "scalars": 0}); ea.Reload()
    t = ea.Tags()
    if tag in t["scalars"]:
        ev = ea.Scalars(tag); return [(e.step, e.value) for e in ev]
    if tag in t["tensors"]:
        out = []
        for e in ea.Tensors(tag):
            a = make_ndarray(e.tensor_proto)
            if a.ndim == 0 and a.dtype.kind in "fi": out.append((e.step, float(a)))
        return out
    return None

def last(d, tag):
    s = series(d, tag); return s[-1][1] if s else np.nan

TEST = ["test_final/accuracy","test_final/macro_f1","test_final/phone_accuracy",
        "test_final/watch_accuracy","generalization/loss_gap"]
rows = []
folds = glob.glob(os.path.join(log_root, "run_*", "*", "fold_*"))
for f in sorted(folds):
    p = f.replace(log_root, "").strip(os.sep).split(os.sep)
    r = {"run": p[0], "fold": p[2]}
    tr, va = os.path.join(f, "train"), os.path.join(f, "validation")
    for name, d in [("train", tr), ("val", va)]:
        for m in ["accuracy", "loss"]:
            s = series(d, f"epoch_{m}")
            if not s: continue
            v = np.array([x[1] for x in s])
            r[f"{name}_{m}_final"] = v[-1]
            r[f"{name}_{m}_best"] = v.max() if m == "accuracy" else v.min()
            r[f"{name}_{m}_best_epoch"] = int(v.argmax() if m == "accuracy" else v.argmin())
        s = series(d, "epoch_accuracy")
        if s: r[f"{name}_epochs"] = len(s)
    r["lr_final"] = last(tr, "epoch_learning_rate")
    for t in TEST:
        r[t.split("/")[-1] if "test_final" in t else t] = last(f, t)
    rows.append(r)

df = pd.DataFrame(rows)
df["val_train_loss_gap"] = df["val_loss_final"] - df["train_loss_final"]
pd.set_option("display.width", 250, "display.max_columns", 50)
print(df.groupby("run").size())
print(df.round(4).to_string(index=False))

num = df.drop(columns="fold").groupby("run").agg(["mean", "std"]).round(4)
print(num.T.to_string())
df.to_csv("all_runs_stats.csv", index=False)

run
run_2026-11-26_03-08-26    9
run_2026-11-26_05-45-18    9
run_2026-11-26_07-35-42    9
run_2026-11-26_11-19-50    9
run_2026-11-26_12-44-19    9
run_2026-11-26_13-30-43    9
dtype: int64
                    run              fold  train_accuracy_final  train_accuracy_best  train_accuracy_best_epoch  train_loss_final  train_loss_best  train_loss_best_epoch  train_epochs  val_accuracy_final  val_accuracy_best  val_accuracy_best_epoch  val_loss_final  val_loss_best  val_loss_best_epoch  val_epochs  lr_final  accuracy  macro_f1  phone_accuracy  watch_accuracy  generalization/loss_gap  val_train_loss_gap
run_2026-11-26_03-08-26 fold_test_a_val_b                0.9396               0.9396                         10            0.2008           0.2008                     10            11              0.8863             0.9092                        5          0.3142         0.2476                    5          11    0.0000    0.6889    0.7120          0.6829          0.7233                 

In [8]:
import pandas as pd

meta = pd.DataFrame({"participant": User, "device": device_ids})
meta["is_watch"] = meta["device"].astype(str).str.contains("gear|lgwatch", case=False)

print(pd.crosstab(meta["participant"], meta["device"]))
print(meta.groupby("participant")["is_watch"].agg(watch_windows="sum", total="size"))

NameError: name 'User' is not defined

In [9]:
import pandas as pd

meta = pd.read_csv(r"<folder with X.npy>\metadata.csv")
print(meta.columns.tolist())
print(len(meta), len(y))          # both should be 37192

user_col  = next(c for c in meta.columns if c.lower() in ("user", "participant", "subject"))
model_col = next(c for c in meta.columns if c.lower() in ("model", "device_family", "family"))

meta["participant"] = meta[user_col]
meta["family"] = meta[model_col].astype(str)
meta["is_watch"] = meta["family"].str.lower().isin(["gear", "lgwatch"])

print(pd.crosstab(meta["participant"], meta["family"]))
print(meta.groupby("participant")["is_watch"].agg(watch_windows="sum", total="size"))

OSError: [Errno 22] Invalid argument: '<folder with X.npy>\\metadata.csv'